# 🧹 AgentCore End-to-End Cleanup

This notebook provides a comprehensive cleanup process for all resources created during the AgentCore End-to-End tutorial.

## Overview

This cleanup process will remove:
- **Memory**: AgentCore Memory resources and stored data
- **Runtime**: Agent runtime instances and ECR repositories
- **Security**: Execution roles, and Authorization Provider resources
- **Observability**: CloudWatch log groups and streams
- **Local Files**: Generated configuration and code files

⚠️ **Important**: This cleanup is irreversible. Make sure you have saved any important data (if needed) before proceeding.

---

## Step 1: Import Required Dependencies

Load all necessary modules and helper functions for the cleanup process.

In [ ]:
import boto3
import os
from botocore.exceptions import ClientError

from bedrock_agentcore_starter_toolkit import Runtime
from lab_helpers.lab2_memory import delete_memory, REGION
from lab_helpers.utils import (
    delete_agentcore_runtime_execution_role,
    delete_ssm_parameter,
    cleanup_cognito_resources,
    get_customer_support_secret,
    delete_customer_support_secret
)

print("✅ Dependencies imported successfully")
print(f"🌍 Working in region: {REGION}")

## Step 2: Clean Up Memory Resources

Remove AgentCore Memory resources and associated data.

In [ ]:
print("🧠 Starting Memory cleanup...")

try:
    # Note: memory_hooks should be available from previous lab sessions
    # TODO: recreate the memory hooks
    delete_memory(memory_hooks)
    print("  ✅ Memory resources cleaned up successfully")
except NameError:
    print("  ℹ️  memory_hooks not found - may have been cleaned up already or not created")
except Exception as e:
    print(f"  ⚠️  Error cleaning up memory: {e}")

## Step 3: Clean Up Runtime Resources

Remove the AgentCore Runtime, ECR repository, and associated AWS resources.

In [ ]:
print("🚀 Starting Runtime cleanup...")

try:
    # Initialize runtime and get launch result
    agentcore_runtime = Runtime()
    launch_result = agentcore_runtime.launch()
    
    # Initialize AWS clients
    agentcore_control_client = boto3.client("bedrock-agentcore-control", region_name=REGION)
    ecr_client = boto3.client("ecr", region_name=REGION)
    
    # Delete the AgentCore Runtime
    print("  🗑️  Deleting AgentCore Runtime...")
    response = agentcore_control_client.delete_agent_runtime(
        agentRuntimeId=launch_result.agent_id
    )
    print(f"  ✅ Agent runtime deleted: {response['status']}")
    
    # Delete the ECR repository
    print("  🗑️  Deleting ECR repository...")
    repository_name = launch_result.ecr_uri.split("/")[1]
    response = ecr_client.delete_repository(
        repositoryName=repository_name, force=True
    )
    print(f"  ✅ ECR repository deleted: {response['repository']['repositoryName']}")
    
except Exception as e:
    print(f"  ⚠️  Error during runtime cleanup: {e}")

## Step 4: Clean Up Security Resources

Remove guardrails, execution roles, and authentication resources.

In [ ]:
print("🛡️  Starting Security cleanup...")

try:
    bedrock_client = boto3.client("bedrock", region_name=REGION)
    
    # Delete execution role
    print("  🗑️  Deleting AgentCore Runtime execution role...")
    delete_agentcore_runtime_execution_role()
    print("  ✅ Execution role deleted")
    
    # Delete SSM parameter
    print("  🗑️  Deleting SSM parameter...")
    delete_ssm_parameter("/app/customersupport/agentcore/runtime_arn")
    print("  ✅ SSM parameter deleted")
    
    # Clean up Cognito and secrets
    print("  🗑️  Cleaning up Cognito resources...")
    cs = get_customer_support_secret()
    cleanup_cognito_resources(cs['pool_id'])
    print("  ✅ Cognito resources cleaned up")
    
    print("  🗑️  Deleting customer support secret...")
    delete_customer_support_secret()
    print("  ✅ Customer support secret deleted")
    
except Exception as e:
    print(f"  ⚠️  Error during security cleanup: {e}")

## Step 5: Clean Up Local Files

Remove generated configuration and code files from the local directory.

In [ ]:
print("📁 Starting Local Files cleanup...")

# List of files to clean up
files_to_delete = [
    "Dockerfile",
    ".dockerignore",
    ".bedrock_agentcore.yaml",
    "customer_support_agent.py",
    "agent_runtime.py",
]

deleted_files = []
missing_files = []

for file in files_to_delete:
    if os.path.exists(file):
        try:
            os.unlink(file)
            deleted_files.append(file)
            print(f"  ✅ Deleted {file}")
        except Exception as e:
            print(f"  ⚠️  Error deleting {file}: {e}")
    else:
        missing_files.append(file)

if deleted_files:
    print(f"\n📁 Successfully deleted {len(deleted_files)} files")
if missing_files:
    print(f"ℹ️  {len(missing_files)} files were already missing: {', '.join(missing_files)}")

## Step 6: Clean Up Observability Resources

Remove CloudWatch log groups and streams used for agent monitoring.

In [ ]:
print("📊 Starting Observability cleanup...")

# Configuration
log_group_name = "agents/customer-support-assistant-logs"
log_stream_name = "default"

logs_client = boto3.client("logs", region_name=REGION)

# Delete log stream first (must be done before deleting log group)
try:
    print(f"  🗑️  Deleting log stream '{log_stream_name}'...")
    logs_client.delete_log_stream(
        logGroupName=log_group_name, logStreamName=log_stream_name
    )
    print(f"  ✅ Log stream '{log_stream_name}' deleted successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print(f"  ℹ️  Log stream '{log_stream_name}' doesn't exist")
    else:
        print(f"  ⚠️  Error deleting log stream: {e}")

# Delete log group
try:
    print(f"  🗑️  Deleting log group '{log_group_name}'...")
    logs_client.delete_log_group(logGroupName=log_group_name)
    print(f"  ✅ Log group '{log_group_name}' deleted successfully")
except ClientError as e:
    if e.response["Error"]["Code"] == "ResourceNotFoundException":
        print(f"  ℹ️  Log group '{log_group_name}' doesn't exist")
    else:
        print(f"  ⚠️  Error deleting log group: {e}")

## 🎉 Cleanup Complete!

All AgentCore resources have been cleaned up. Here's a summary of what was removed:

In [ ]:
print("\n" + "=" * 60)
print("🧹 CLEANUP COMPLETED SUCCESSFULLY! 🧹")
print("=" * 60)
print()
print("📋 Resources cleaned up:")
print("  🧠 Memory: AgentCore Memory resources and data")
print("  🚀 Runtime: Agent runtime and ECR repository")
print("  🛡️ Security: Roles, and SSM secrets")
print("  📊 Observability: CloudWatch logs")
print("  📁 Files: Local configuration files")
print()
print("✨ Your AWS account is now clean and ready for new experiments!")
print("\nThank you for completing the AgentCore End-to-End tutorial! 🚀")